In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

In [3]:
paths_my = {
    "2030": "pIRF/pirf_median_by_ssp-all_H200_MY2030.xlsx",
    "2040": "pIRF/pirf_median_by_ssp-all_H200_MY2040.xlsx",
    "2050": "pIRF/pirf_median_by_ssp-all_H200_MY2050.xlsx",
}

paths_est = {
    "CO2": "to_comp_withWatanabe_paper/pIRF_CO2.xlsx",
    "CH4": "to_comp_withWatanabe_paper/pIRF_CH4.xlsx",
    "N2O": "to_comp_withWatanabe_paper/pIRF_N2O.xlsx",
}

mapping = [
    ("RCP2.6", "ssp126"),
    ("RCP4.5", "ssp245"),
    ("RCP6.0", "ssp460"),
    ("RCP8.5", "ssp585"),
]

def detect_cols(df):
    df.columns = [str(c) for c in df.columns]
    col_gas = [c for c in df.columns if c.lower() == "gas"][0]
    col_ssp = [c for c in df.columns if "ssp" in c.lower() or "scenario" in c.lower()][0]
    col_h = [c for c in df.columns if c.lower() in ["h","horizon","time","t","year"]][0]
    col_val = [c for c in df.columns if c.lower() in ["value","median","pirf"]][0]
    return col_gas, col_ssp, col_h, col_val

# Load MY data
my_data = {}
for year, path in paths_my.items():
    df = pd.read_excel(path, sheet_name="median_long")
    my_data[year] = dict(zip(
        ["df","gas","ssp","h","val"],
        (df, *detect_cols(df))
    ))


In [14]:
out_dir = "pIRF/plot_compare_Watanabe"
os.makedirs(out_dir, exist_ok=True)

panel_paths = []

for gas in ["CO2","CH4","N2O"]:
    for rcp, ssp in mapping:
        
        plt.figure()
        
        # ----- FaIR lines -----
        if gas == "CO2":
            styles = {"2030":"-","2040":"--","2050":":"}
            for year in ["2030","2040","2050"]:
                df = my_data[year]["df"]
                gcol = my_data[year]["gas"]
                scol = my_data[year]["ssp"]
                hcol = my_data[year]["h"]
                vcol = my_data[year]["val"]
                
                df_sel = df[(df[gcol]==gas) & (df[scol].str.lower()==ssp)]
                df_sel = df_sel.sort_values(hcol)
                
                plt.plot(
                    df_sel[hcol], df_sel[vcol],
                    linestyle=styles[year],
                    color="blue", #lightblue
                    label=f"FaIR - MY{year}, {ssp}"
                )
        else:
            df = my_data["2030"]["df"]
            gcol = my_data["2030"]["gas"]
            scol = my_data["2030"]["ssp"]
            hcol = my_data["2030"]["h"]
            vcol = my_data["2030"]["val"]
            
            df_sel = df[(df[gcol]==gas) & (df[scol].str.lower()==ssp)]
            df_sel = df_sel.sort_values(hcol)
            
            plt.plot(
                df_sel[hcol], df_sel[vcol],
                linestyle="-",
                color="blue", #lightblue
                label=f"FaIR - MY2030, {ssp}"
            )
        
        # ----- Watanabe & Cherubini (2026) -----
        df_est = pd.read_excel(paths_est[gas])
        df_est.columns = [str(c) for c in df_est.columns]
        h_est = [c for c in df_est.columns if c.lower() in ["h","horizon","time","t","year"]][0]
        
        col_est = None
        for c in df_est.columns:
            cl = c.lower().replace(" ","").replace(".","")
            target = rcp.lower().replace("rcp","").replace(".","")
            if target in cl:
                col_est = c
                break
        
        if col_est is None:
            num_cols = [c for c in df_est.select_dtypes(include=[np.number]).columns if c != h_est]
            idx = mapping.index((rcp, ssp))
            col_est = num_cols[idx]
        
        df_est = df_est[[h_est,col_est]].rename(columns={h_est:"H",col_est:"pIRF"})
        df_est = df_est[df_est["H"]<=100]
        
        plt.plot(
            df_est["H"], df_est["pIRF"],
            color="orange",
            linestyle="-",
            label=f"Watanabe and Cherubini (2026), {rcp}"
        )
        
        # Title only gas name
        plt.title(f"{gas}")
        plt.xlabel("Time since pulse (years)")
        plt.ylabel("decay factor (prospective Impulse Response)")
        plt.legend(fontsize=12)
        
        fname = f"{gas}_{rcp.replace('.','p')}.png"
        fpath = os.path.join(out_dir,fname)
        plt.savefig(fpath,dpi=300,bbox_inches="tight")
        plt.close()
        
        panel_paths.append(fpath)


# ---- 3x4 grid ----
ordered = []
for gas in ["CO2","CH4","N2O"]:
    for rcp,_ in mapping:
        ordered.append(os.path.join(out_dir,f"{gas}_{rcp.replace('.','p')}.png"))

imgs = [Image.open(p).convert("RGB") for p in ordered]

w = max(im.size[0] for im in imgs)
h = max(im.size[1] for im in imgs)

def pad(im,w,h):
    bg = Image.new("RGB",(w,h),(255,255,255))
    bg.paste(im,((w-im.size[0])//2,(h-im.size[1])//2))
    return bg

imgs = [pad(im,w,h) for im in imgs]

cols = 4
rows = 3
grid = Image.new("RGB",(w*cols,h*rows),(255,255,255))

for i,im in enumerate(imgs):
    r = i//cols
    c = i%cols
    grid.paste(im,(c*w,r*h))

final_path = os.path.join(out_dir, "pIRF_final_3gas_4scn_wo_uncertainty.png")
grid.save(final_path, "PNG")

print(f"Saved to: {final_path}")

Saved to: pIRF/plot_compare_Watanabe/pIRF_final_3gas_4scn_wo_uncertainty.png


### now adding uncertainty and save 

In [17]:
import pickle

paths_my_ens = {
    "2030": "pIRF/pirf_by_ssp-all_H200_MY2030_ens.pkl",
    "2040": "pIRF/pirf_by_ssp-all_H200_MY2040_ens.pkl",
    "2050": "pIRF/pirf_by_ssp-all_H200_MY2050_ens.pkl",
}

with open(paths_my_ens[BAND_MY], "rb") as f:
    pirf_ens_band = pickle.load(f)   # dict[ssp][gas] -> (H, 1001)

In [22]:
# now to add uncertinatly::  ModelYear with uncertainty band?
BAND_MY = "2030"          # caz we only plot 2030 for CH4/N2O, default is 2030   "2030", "2040", "2050"
BAND_Q_LO, BAND_Q_HI = 0.01, 0.99   # e.g., 5–95% band  
BAND_N = 1001

In [25]:
out_dir = "pIRF/plot_compare_Watanabe_wh1001"
os.makedirs(out_dir, exist_ok=True)

panel_paths = []

for gas in ["CO2","CH4","N2O"]:
    for rcp, ssp in mapping:
        
        plt.figure()
        
        # ----- FaIR lines -----
        if gas == "CO2":
            styles = {"2030":"-","2040":"--","2050":":"}
            for year in ["2030","2040","2050"]:
                df = my_data[year]["df"]
                gcol = my_data[year]["gas"]
                scol = my_data[year]["ssp"]
                hcol = my_data[year]["h"]
                vcol = my_data[year]["val"]
                
                df_sel = df[(df[gcol]==gas) & (df[scol].str.lower()==ssp)]
                df_sel = df_sel.sort_values(hcol)
                
                plt.plot(
                    df_sel[hcol], df_sel[vcol],
                    linestyle=styles[year],
                    color="lightblue",
                    label=f"FaIR - MY{year}, {ssp}"
                )
        else:
            df = my_data["2030"]["df"]
            gcol = my_data["2030"]["gas"]
            scol = my_data["2030"]["ssp"]
            hcol = my_data["2030"]["h"]
            vcol = my_data["2030"]["val"]
            
            df_sel = df[(df[gcol]==gas) & (df[scol].str.lower()==ssp)]
            df_sel = df_sel.sort_values(hcol)
            
            plt.plot(
                df_sel[hcol], df_sel[vcol],
                linestyle="-",
                color="lightblue",
                label=f"FaIR - MY2030, {ssp}"
            )

        # ---- Add uncertainty band for the selected ModelYear only ----
        df_band = my_data[BAND_MY]["df"]
        gcol_b = my_data[BAND_MY]["gas"]
        scol_b = my_data[BAND_MY]["ssp"]
        hcol_b = my_data[BAND_MY]["h"]
        vcol_b = my_data[BAND_MY]["val"]
        
        df_sel_band = df_band[(df_band[gcol_b] == gas) & (df_band[scol_b].str.lower() == ssp)].copy()
        df_sel_band = df_sel_band.sort_values(hcol_b)
        
        H_band = df_sel_band[hcol_b].to_numpy()
        
        # ensemble array: expected shape (len(H), 1001)
        ens = pirf_ens_band[ssp][gas]
        
        # safety: slice if needed
        ens = ens[:len(H_band), :]
        
        q_lo = np.quantile(ens, BAND_Q_LO, axis=1)
        q_hi = np.quantile(ens, BAND_Q_HI, axis=1)
        
        band_label = (
            f"FaIR uncertainty: MY{BAND_MY}, {ssp} "
            f"q{int(BAND_Q_LO*100):02d}–q{int(BAND_Q_HI*100):02d}" #(N={BAND_N})
        )

        # band AND legend 
        plt.fill_between(H_band, q_lo, q_hi, alpha=0.15, linewidth=0, label=band_label)    



        
        # ----- Watanabe & Cherubini (2026) -----
        df_est = pd.read_excel(paths_est[gas])
        df_est.columns = [str(c) for c in df_est.columns]
        h_est = [c for c in df_est.columns if c.lower() in ["h","horizon","time","t","year"]][0]
        
        col_est = None
        for c in df_est.columns:
            cl = c.lower().replace(" ","").replace(".","")
            target = rcp.lower().replace("rcp","").replace(".","")
            if target in cl:
                col_est = c
                break
        
        if col_est is None:
            num_cols = [c for c in df_est.select_dtypes(include=[np.number]).columns if c != h_est]
            idx = mapping.index((rcp, ssp))
            col_est = num_cols[idx]
        
        df_est = df_est[[h_est,col_est]].rename(columns={h_est:"H",col_est:"pIRF"})
        df_est = df_est[df_est["H"]<=100]
        
        plt.plot(
            df_est["H"], df_est["pIRF"],
            color="orange",
            linestyle="-",
            label=f"Watanabe and Cherubini (2026), {rcp}"
        )
        
        # Title only gas name
        plt.title(f"{gas}")
        plt.xlabel("Time since pulse (years)")
        plt.ylabel("decay factor (prospective Impulse Response)")
        plt.legend(fontsize=12)
        
        fname = f"{gas}_{rcp.replace('.','p')}.png"
        fpath = os.path.join(out_dir,fname)
        plt.savefig(fpath,dpi=300,bbox_inches="tight")
        plt.close()
        
        panel_paths.append(fpath)



# ---- 3x4 grid ----
ordered = []
for gas in ["CO2","CH4","N2O"]:
    for rcp,_ in mapping:
        ordered.append(os.path.join(out_dir,f"{gas}_{rcp.replace('.','p')}.png"))

imgs = [Image.open(p).convert("RGB") for p in ordered]

w = max(im.size[0] for im in imgs)
h = max(im.size[1] for im in imgs)

def pad(im,w,h):
    bg = Image.new("RGB",(w,h),(255,255,255))
    bg.paste(im,((w-im.size[0])//2,(h-im.size[1])//2))
    return bg

imgs = [pad(im,w,h) for im in imgs]

cols = 4
rows = 3
grid = Image.new("RGB",(w*cols,h*rows),(255,255,255))

for i,im in enumerate(imgs):
    r = i//cols
    c = i%cols
    grid.paste(im,(c*w,r*h))

final_path = os.path.join(out_dir, "pIRF_wh_1001.png")
grid.save(final_path, "PNG")

print(f"Saved to: {final_path}")


Saved to: pIRF/plot_compare_Watanabe_wh1001/pIRF_wh_1001.png
